# Predicting the next word: from a bigram model to Transformers

**Course: Artificial Intelligence · Master 2 · Chapter 3, deep learning: text generation with Transformers**

Notebook 08 · duration: about 1 h 30 · data: 20 sentences invented for this notebook and six public-domain quotations

**What you will do**
- build, in plain Python and with no internet, a small language model that predicts the next word from 20 sentences about reading, then make it write sentences;
- adjust the **temperature** of the draw and watch the text become predictable, then surprising, then incoherent;
- try real Transformers (online), then draw up your own checklist for using a chatbot in your language and literature studies.

**How to use this notebook**
Run each cell from top to bottom: click the ▶ button at its left, or press Shift+Enter. You never have to write code. Change only the fields of the cells titled "TO CHANGE". In Colab, the menu **Runtime → Run all** runs the whole notebook at once.

Before each cell, write down your **prediction**; afterwards, compare with "What you should see". The cells in Part 3 need an internet connection (they work in Colab); if you are offline, read them and go on to Part 4: nothing after them depends on them.

| Part | Content | Time |
|---|---|---|
| 1 | A hand-made bigram model | 35 min |
| 2 | Temperature | 15 min |
| 3 | Real Transformers (online) | 15 min |
| 4 | Using a chatbot critically | 15 min |
| Wrap-up | Your turn, to hand in | 10 min |

## Part 1 · A hand-made language model

A **language model** answers a single question: *which word comes next?* Your phone keyboard does it when it offers three words above the keys; GPT does it too, token after token (chapter 3, "Text generation with Transformers").

We are going to build the simplest one of all, the **bigram model**. A bigram is a pair of neighbouring words: "we read", "read a". The model counts these pairs in a corpus, then uses them to predict and to write.

The corpus: 20 short sentences about reading, books, the library and the classroom, **invented for this notebook** (made-up data), in lower case and without punctuation to keep things simple. The field `your_sentence` lets you add one: leave it empty for now.

**Before running:** 20 sentences of 4 to 10 words… how many words in total (tokens), in your opinion? And how many different words?

In [ ]:
#@title TO CHANGE (optional): the corpus, plus your own sentence
corpus = """we read a novel in the library
we read a poem in class
we read a book at night
the library opens in the morning
the library lends books to students
the teacher reads a poem in class
the teacher marks the essays at night
the students read a novel in class
the students borrow books from the library
a novel tells a story
a poem tells a secret
the librarian puts the books on the shelf
the class listens to the teacher
the book is on the table
the novel is long but the poem is short
my sister reads a novel on the train
we read the poem aloud
the dictionary helps the students in class
every book tells a story
reading opens the mind"""
your_sentence = "" #@param {type:"string"}

sentences = corpus.split("\n")                  # one sentence per line
if your_sentence != "":
    sentences.append(your_sentence.lower().strip(" .!?"))
words = " ".join(sentences).split()             # all the words, one after the other
print("Number of sentences:", len(sentences))
print("Number of words (tokens):", len(words))
print("Number of different words:", len(set(words)))   # set() keeps one copy of each word

**What you should see**
```
Number of sentences: 20
Number of words (tokens): 127
Number of different words: 46
```
(A little more if you added a sentence.) As in chapter 2, we distinguish **tokens** (each occurrence of a word) and **types** (the different words). Our model will only ever know these 46 words.

**Question:** a model trained on this corpus will talk mostly about books and classes. Why? What would it say if it had been trained on cooking recipes?

### Counting pairs of neighbouring words

For each word, we write down the word that follows it, and we count. We add a full stop "." at the end of each sentence: for the model it is a word like any other, meaning "end of sentence". So "we read a book at night" gives six pairs: we → read, read → a, a → book, book → at, at → night, night → ".".

The result is a dictionary (chapter 2): each word is linked to a `Counter` of the words that follow it. The model **learns nothing else** but these counts.

**Before running:** read the corpus again. Which words follow "library", and how many times each?

In [ ]:
#@title Count, for each word, the words that follow it (the bigrams)
from collections import Counter

def count_next_words(list_of_sentences):
    next_words = {}                                  # word -> Counter of the next words
    for sentence in list_of_sentences:
        sentence_words = sentence.lower().split() + ["."]   # the full stop marks the end
        for i in range(len(sentence_words) - 1):
            word, following = sentence_words[i], sentence_words[i + 1]
            if word not in next_words:
                next_words[word] = Counter()
            next_words[word][following] += 1
    return next_words

next_words = count_next_words(sentences)
print("Words that have at least one next word:", len(next_words))
print('After "library":', next_words["library"].most_common())

**What you should see**
```
Words that have at least one next word: 46
After "library": [('.', 2), ('opens', 1), ('lends', 1)]
```
"library" ends two sentences (hence the full stop, twice) and is followed once by "opens" and once by "lends". All 46 words have a next word, since every sentence ends with a full stop. Altogether the model has kept **89 different pairs**: these are all its "parameters".

**Question:** does this model "know" what a library is? What exactly does it know?

### The table of next words, and a prediction

Choose a word in the menu (or type one). The cell shows the table of the words that follow it, with their **count** and their **probability**: the count divided by the total. The most likely next word is the one with the highest count; in case of a tie, Python keeps the first one met in the corpus.

**Before running:** after "a", which word comes most often? With roughly what probability?

In [ ]:
#@title TO CHANGE: the word to study
chosen_word = "a" #@param ["a", "the", "we", "read", "tells", "is"] {allow-input: true}

counter = next_words.get(chosen_word, Counter())   # empty Counter if the word is unknown
total = sum(counter.values())
print(f'"{chosen_word}" is followed {total} times in the corpus.')
print("   next word      count      probability")
for following, n in counter.most_common():
    print(f"   {following:15} {n:5}      {n / total:.2f}")
if total > 0:
    print("Prediction: the most likely next word is", counter.most_common(1)[0][0])

**What you should see**
```
"a" is followed 11 times in the corpus.
   next word      count      probability
   novel               4      0.36
   poem                3      0.27
   story               2      0.18
   book                1      0.09
   secret              1      0.09
Prediction: the most likely next word is novel
```
The five probabilities add up to 1 (apart from rounding).

**Question:** choose "the" in the menu and run the cell again. Why does "library" win? Is it a property of English or of *this* corpus?

### Writing a random sentence, with a fixed seed

For a language model, writing is a loop: start from a word, **draw** the next one at random *in proportion to its counts* (after "a": "novel" has 4 chances out of 11, "poem" 3, "story" 2, "book" and "secret" 1 each), add it, and start again until the full stop (15 words at most). The draw comes from a pseudo-random generator: with the same **seed**, everyone gets the same sentence; change the seed to get another one.

**Before running:** will the generated sentence appear, word for word, in the corpus? Will it be correct English? Will it make sense?

In [ ]:
#@title TO CHANGE: generate a random sentence (fixed seed)
start_word = "we" #@param ["we", "the", "a", "my", "every", "reading"]
seed = 2 #@param {type:"integer"}
import random

def generate(model, start_word):
    sentence = [start_word]
    while sentence[-1] != "." and len(sentence) < 15:         # until the full stop, 15 words at most
        counter = model[sentence[-1]]
        draw = random.choices(list(counter), weights=list(counter.values()))
        sentence.append(draw[0])                              # draw in proportion to the counts
    return sentence

random.seed(seed)                          # same seed, same sentence for everyone
sentence = generate(next_words, start_word)
print(" ".join(sentence))
print("Is this sentence in the corpus?", " ".join(sentence[:-1]) in sentences)

**What you should see**
```
we read the library .
Is this sentence in the corpus? False
```
A **new** sentence (`False`: it is not in the corpus), apparently well formed… and absurd: you cannot read a library.

**Question:** try seeds 5, 6 and 36. Which ones give a sentence that is correct and sensible, correct but absurd, or incorrect?

### Checking the sentence, pair by pair

**Before running:** has each pair of neighbouring words in the generated sentence already been seen in the corpus? If so, how many times?

In [ ]:
#@title Check each pair of the generated sentence
for i in range(len(sentence) - 1):
    word, following = sentence[i], sentence[i + 1]
    print(f"{word:>14} → {following:14} count in the corpus: {next_words[word][following]}")

**What you should see**
```
            we → read           count in the corpus: 4
          read → the            count in the corpus: 1
           the → library        count in the corpus: 4
       library → .              count in the corpus: 2
```
Every pair exists in the corpus: locally, everything is "right". "read the" comes from "we read the poem aloud", "the library" from four other sentences. It is the combination that makes no sense.

**Question:** at which point in the sentence did the model "forget" what "we" were reading?

### Why it sounds right but means nothing

- **Only one word of context.** The model looks only at the previous word. After "read the", it no longer knows who was reading or what.
- **No knowledge of the world.** It has never seen a teacher or a novel, only strings of characters. It knows *co-occurrences*, not *meanings*. The linguist J. R. Firth said: "You shall know a word by the company it keeps"; our model knows only the company.
- **No intention.** It does not mean anything: it chains together what is plausible according to its counts.
- **It invents.** By recombining pieces it has seen, it produces sentences that exist nowhere: this is its apparent creativity, and its danger.

Large models such as GPT follow the same principle, predicting the next token, but with thousands of tokens of context, the attention mechanism and billions of parameters. Their sentences sound far more right and meaning *seems* to be there; their goal is still plausibility, not truth. Hence hallucinations (Part 4).

## Part 2 · Temperature

So far, a word seen 3 times had 3 times more chances of being drawn than a word seen once. The **temperature** $T$ changes these chances. Chapter 3 gives the formula for the raw scores $z_i$ of a neural network: $p_i = \exp(z_i / T) \,/\, \sum_j \exp(z_j / T)$. For our counts $n_i$ (taking $z_i = \ln n_i$), it amounts to raising each count to the power $1/T$, then dividing by the total:

$$p_i = \frac{n_i^{1/T}}{\sum_j n_j^{1/T}}$$

- $T = 1$: the probabilities stay proportional to the counts;
- $T < 1$: the gaps widen, and the most frequent word takes almost everything (predictable text);
- $T > 1$: the gaps flatten out, and rare words get their chance (surprising, then incoherent text).

**Before running:** after "the", "library" has 4 chances out of 24, i.e. 0.17, at $T = 1$. At $T = 0.3$, will its probability go up or down? How far?

In [ ]:
#@title TO CHANGE: a word and a temperature
studied_word = "the" #@param ["the", "a", "read", "in"]
temperature = 0.3 #@param {type:"slider", min:0.1, max:3, step:0.1}

def probabilities(counter, temperature):
    weights = {}
    for word, n in counter.items():
        weights[word] = n ** (1 / temperature)       # count "to the power 1/T"
    total = sum(weights.values())
    return {word: weights[word] / total for word in weights}  # they add up to 1

proba_1 = probabilities(next_words[studied_word], 1)
proba_t = probabilities(next_words[studied_word], temperature)
print(f'After "{studied_word}"     count    T = 1   T = {temperature}')
for word, n in next_words[studied_word].most_common():
    print(f"   {word:15} {n:5}    {proba_1[word]:.2f}    {proba_t[word]:.2f}")

**What you should see**
```
After "the"     count    T = 1   T = 0.3
   library             4    0.17    0.50
   teacher             3    0.12    0.19
   students            3    0.12    0.19
   poem                2    0.08    0.05
   morning             1    0.04    0.00
   essays              1    0.04    0.00
   librarian           1    0.04    0.00
   books               1    0.04    0.00
   shelf               1    0.04    0.00
   class               1    0.04    0.00
   book                1    0.04    0.00
   table               1    0.04    0.00
   novel               1    0.04    0.00
   train               1    0.04    0.00
   dictionary          1    0.04    0.00
   mind                1    0.04    0.00
```
At $T = 0.3$, "library" goes from 0.17 to 0.50: it takes half of the draws, and the twelve words seen only once fall to almost nothing (0.00 once rounded).

**Question:** push the slider to 3 and run the cell again. What happens to the gaps between words? Which of the two settings is closer to a "completely random" choice?

### Three temperatures side by side

The chart shows the same next-word distribution at three temperatures: 0.3, 1 and 3. The words are sorted from the most frequent (at the top) to the rarest.

**Before running:** what will the bar for "library" look like in the right-hand panel ($T = 3$)?

In [ ]:
#@title Plot the distribution at T = 0.3, 1 and 3
import matplotlib.pyplot as plt
following_words = [word for word, n in next_words[studied_word].most_common()]
fig, axes = plt.subplots(1, 3, figsize=(11, 1.5 + 0.35 * len(following_words)), sharey=True, sharex=True)
for ax, t in zip(axes, [0.3, 1, 3]):
    proba = probabilities(next_words[studied_word], t)
    ax.barh(following_words, [proba[word] for word in following_words], color="#2a78d6", height=0.7, zorder=3)
    ax.set_title(f"T = {t}")
    ax.set_xlabel("probability")
    ax.grid(axis="x", color="#e5e5e5")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].invert_yaxis()                     # most frequent word at the top
fig.suptitle(f'After "{studied_word}": the next word depending on the temperature')
plt.tight_layout()
plt.show()

**What you should see**

Three panels of horizontal bars. On the left ($T = 0.3$), one long bar, "library" (0.50), two medium bars, "teacher" and "students" (0.19), and many bars that are almost invisible. In the middle ($T = 1$), the proportions of the corpus. On the right ($T = 3$), nearly equal bars, between 0.06 and 0.09: rare words are almost as likely as frequent ones.

**Question:** if you had to describe temperature to a literature classmate without a formula, which image or metaphor would you choose?

### Drawing the next word twenty times

A probability is easier to see on actual draws. The cell draws the word that follows the studied word 20 times, at three temperatures, with a fixed seed.

**Before running:** out of 20 draws at $T = 0.2$, how many times "library", in your opinion? And at $T = 3$?

In [ ]:
#@title Draw the next word 20 times, at three temperatures
random.seed(1)
for t in [0.2, 1, 3]:
    proba = probabilities(next_words[studied_word], t)
    draws = random.choices(list(proba), weights=list(proba.values()), k=20)
    print(f"T = {t}:", Counter(draws).most_common())

**What you should see**
```
T = 0.2: [('library', 12), ('students', 4), ('teacher', 4)]
T = 1: [('teacher', 6), ('students', 5), ('library', 3), ('librarian', 1), ('dictionary', 1), ('poem', 1), ('books', 1), ('class', 1), ('morning', 1)]
T = 3: [('train', 3), ('students', 2), ('novel', 2), ('poem', 2), ('class', 2), ('mind', 1), ('morning', 1), ('dictionary', 1), ('books', 1), ('table', 1), ('shelf', 1), ('library', 1), ('essays', 1), ('librarian', 1)]
```
At $T = 0.2$, "library" comes out 12 times out of 20 and only three words ever appear. At $T = 1$, nine different words. At $T = 3$, fourteen of the sixteen words come out, and "train", seen only once in the corpus, even comes first.

**Question:** why are the counts obtained at $T = 1$ not exactly proportional to those of the corpus (4, 3, 3, 2, 1…)?

### Whole sentences at the chosen temperature

Same loop as in Part 1, but each word is drawn with the probabilities "at temperature $T$". We start from the start word chosen above ("we").

**Before running:** at $T = 0.2$, will the five sentences be different from one another?

In [ ]:
#@title TO CHANGE: five sentences at the chosen temperature
temperature_sentences = 0.2 #@param {type:"slider", min:0.1, max:3, step:0.1}
seed_sentences = 1 #@param {type:"integer"}

random.seed(seed_sentences)
for number in range(1, 6):
    sentence_t = [start_word]
    while sentence_t[-1] != "." and len(sentence_t) < 15:
        proba = probabilities(next_words[sentence_t[-1]], temperature_sentences)
        draw = random.choices(list(proba), weights=list(proba.values()))
        sentence_t.append(draw[0])
    print(number, " ".join(sentence_t))

**What you should see**
```
1 we read a novel in class .
2 we read a novel in class .
3 we read a novel in class .
4 we read a novel in class .
5 we read a novel in class .
```
At $T = 0.2$, the five sentences are identical: the model almost always takes the most frequent word (we → read → a → novel → in → class → .). A low temperature gives predictable, repetitive text.

**Question:** move the slider to 3 and run the cell again. Which temperature would you choose to produce model sentences for a language exercise? And for an "exquisite corpse", the surrealists' writing game?

## Part 3 · Real Transformers (online)

> **Needs an internet connection; works in Colab.** The cells in this part download a model from the Hugging Face Hub (several hundred MB, one or two minutes). If you are offline, read them and go on to Part 4: nothing after them depends on them. On your own computer, first run `pip install transformers torch` (torch is large).

A Transformer does the same thing as our model, predicting a word from the others, but on a completely different scale:

| | Our bigram model | BERT (encoder) | distilgpt2 (decoder) |
|---|---|---|---|
| Task | predict the next word | guess a hidden word (gap-fill) | predict the next token |
| Context | 1 word before | the whole sentence, before **and** after the gap (512 tokens at most) | up to 1,024 tokens before |
| Unit | the word | the token (a word or a piece of a word) | the token |
| What it has learnt | 89 counted pairs | about 110 million parameters | 82 million parameters (124 million for GPT-2) |
| Data | 20 sentences | English books and English Wikipedia | English web pages (an open copy of GPT-2's web corpus, about 40 GB of text) |

Encoders such as BERT read the sentence in both directions: they are trained to recover masked words, exactly like a **cloze test** (gap-fill exercise), well known in language teaching. In a sentence for BERT, the gap is written `[MASK]`. The model `bert-base-uncased` works in lower case.

**Before running:** our model kept 89 pairs. How many parameters will the cell announce for BERT?

In [ ]:
#@title Download BERT, a gap-filling model for English (online)
%pip install -q transformers torch
from transformers import pipeline
gap_filler = pipeline("fill-mask", model="bert-base-uncased")
print("Model ready:", gap_filler.model.name_or_path)
print("Number of parameters:", gap_filler.model.num_parameters())

**What you should see**

After the download (progress bars, sometimes warnings in red: they are normal), two lines: the name of the model, `bert-base-uncased`, and a number of parameters of about 110 million. More than a million times the 89 pairs of our model.

**Question:** where, and on which texts, were these parameters learnt? Which varieties of English does such a corpus represent poorly, in your opinion?

### A gap-fill exercise for BERT

(Online: needs an internet connection; works in Colab.) Choose a sentence in the menu or type your own, with **only one** `[MASK]`. The cell shows the five most likely words for the gap, with their score (a probability between 0 and 1).

**Before running:** first write your own answer. Which word would you put in place of `[MASK]`?

In [ ]:
#@title TO CHANGE: a gap-fill sentence for BERT (online)
gap_sentence = "The librarian puts the [MASK] on the shelves." #@param ["The librarian puts the [MASK] on the shelves.", "Victor Hugo was a French [MASK].", "In class, the students read a [MASK] by Dickens."] {allow-input: true}

for proposal in gap_filler(gap_sentence, top_k=5):
    print(f"{proposal['token_str'].strip():>15}   score {proposal['score']:.2f}")

**What you should see**

Five lines, from the most likely word to the least likely, each with a score between 0 and 1. For the first sentence, expect plausible nouns, most probably "books" or "book" at the top. The exact scores depend on the installed version: write down yours.

**Question:** does the model take into account the words placed *after* the gap ("on the shelves")? Try "The librarian puts the `[MASK]` in the drawer." and compare with our bigram model, which only looked one word back.

### Generating text with distilgpt2

Now a decoder, like GPT: it reads from left to right and predicts the next token, exactly like our bigram model, but with 82 million parameters and up to 1,024 tokens of context. `distilgpt2` is a smaller, faster version of GPT-2 (2019). The starting sentence is the one from chapter 3; the temperature slider works as in Part 2, and the seed makes the draw reproducible on the same installation.

(Online: needs an internet connection; works in Colab. A new download.)

**Before running:** will the two continuations be the same? Will they be grammatical? Will they make sense? Will they be *true*?

In [ ]:
#@title TO CHANGE: text generation with distilgpt2 (online)
start_text = "The old librarian opened the forbidden book and" #@param {type:"string"}
temperature_gpt = 0.8 #@param {type:"slider", min:0.1, max:2, step:0.1}
seed_gpt = 42 #@param {type:"integer"}

from transformers import pipeline, set_seed
generator = pipeline("text-generation", model="distilgpt2")
set_seed(seed_gpt)                         # same seed, same draw (on the same installation)
outputs = generator(start_text, max_new_tokens=30, do_sample=True,
                    temperature=temperature_gpt, num_return_sequences=2)
for output in outputs:
    print(output["generated_text"], "\n")

**What you should see**

Two texts that begin with the starting sentence and continue with about 30 tokens (a little over 20 words). The English is usually grammatical; the story often wanders, repeats itself or contradicts itself. As chapter 3 says, the exact text depends on the seed, the temperature and the installed versions: yours may differ from your neighbour's.

**Question:** is anything in the continuation *true*? Could it be? Try temperatures 0.3 and 1.5: compare with what you saw in Part 2. What has improved since the bigram model, and what has not?

## Part 4 · Using a chatbot critically

A chatbot (ChatGPT and the like) is a very large language model, adjusted for dialogue: it continues your text with the most plausible sequence. **Plausible does not mean true.** Here is the checklist to keep in front of you during your language and literature studies.

1. **Hallucinated references.** A chatbot can give a complete reference (real author, likely title, well-known publisher, year) that does not exist. → Cite only a work you have found in a library catalogue or on the publisher's site, and read yourself.
2. **Invented or distorted quotations.** A "quotation" from Shakespeare or Dickens can be false, altered or attributed to the wrong author. → Check every quotation in the original edition, with the page (for older texts: Project Gutenberg, the Folger Shakespeare Library, Wikisource).
3. **Explanations about language.** Grammar rules, etymology, the meaning of a word: the chatbot answers confidently, sometimes wrongly. → Compare with a reference grammar or a dictionary.
4. **Corrections and translations.** The chatbot can "correct" what was right, erase your style or impose a single variety of the language (American or British English, for example). → Ask for an explanation of each correction; keep only those you understand.
5. **Other people's data and texts.** Do not paste survey answers, learners' work, or unpublished or copyrighted texts.
6. **Declared use.** Declare any use at the end of the work you hand in: tool, date, use, prompts copied in an appendix, how you checked. Handing in a generated text as your own is plagiarism.

These rules are those of the course's usage charter: [Generative AI in your classroom](https://progremer04.github.io/canva-au-for-ang-/en/enseignant.html#guide-ia) (guide, section "A usage charter for the class"). See also the lessons [Outlook, limits and ethics](https://progremer04.github.io/canva-au-for-ang-/en/#c1-perspectives) and [Learning with an AI assistant, without cheating](https://progremer04.github.io/canva-au-for-ang-/en/#me-assistant).

### Where does a fake quotation come from?

Let us go back to our bigram model, trained this time on six real public-domain quotations: Shakespeare (*Hamlet*, around 1600), Keats (*Endymion*, 1818), Dickens (*A Tale of Two Cities*, 1859), Lincoln (Gettysburg Address, 1863), Blake ("The Tyger", 1794) and Emily Dickinson (a poem published in 1891). We start from the first word of one author's quotation, let the model continue, then attribute the result to that author, as an over-confident answer would.

**Before running:** will the "quotation" we get exist? Will it sound like Keats?

In [ ]:
#@title TO CHANGE: make a fake quotation
quotations = {"William Shakespeare": "to be or not to be that is the question",
              "John Keats": "a thing of beauty is a joy for ever",
              "Charles Dickens": "it was the best of times it was the worst of times",
              "Abraham Lincoln": "government of the people by the people for the people shall not perish from the earth",
              "William Blake": "tyger tyger burning bright in the forests of the night",
              "Emily Dickinson": "hope is the thing with feathers that perches in the soul"}   # six real quotations, public domain
author = "John Keats" #@param ["William Shakespeare", "John Keats", "Charles Dickens", "Abraham Lincoln", "William Blake", "Emily Dickinson"]
seed_quotation = 8 #@param {type:"integer"}

quotation_model = count_next_words(list(quotations.values()))
random.seed(seed_quotation)
fake = generate(quotation_model, quotations[author].split()[0])
text = " ".join(fake[:-1])
print(f'"{text.capitalize()}" ({author})')
print("Does this quotation really exist?", text in quotations.values())

**What you should see**
```
"A thing with feathers that is the soul" (John Keats)
Does this quotation really exist? False
```
A line that sounds like poetry, signed with a real name… that Keats never wrote. Only "a thing" comes from Keats; "thing with feathers that" and "the soul" come from Emily Dickinson, and "that is the" from *Hamlet*. Every pair of words is genuine; the whole is false, and it even gives Dickinson's words to the wrong poet. A large model does the same thing, far more convincingly: this is how invented quotations and references are born.

**Question:** without the last line, how could you have exposed this fake quotation? Where would you go to check it?

### Your checklist and your declaration of use

Think of the last time you used a chatbot for your studies. Fill in the fields (tool, use, check) and tick only what you **really** did. The cell shows your checklist and a declaration of use modelled on the guide, to copy at the end of your work.

**Before running:** how many boxes will you be able to tick honestly?

In [ ]:
#@title TO CHANGE: my checklist and my declaration of use
tool = "a chatbot (free online version, October 2026)" #@param {type:"string"}
use = "have the notion of temperature explained to me" #@param {type:"string"}
check = "I compared the answer with the chapter 3 lesson" #@param {type:"string"}
references_found = False #@param {type:"boolean"}
quotations_checked = False #@param {type:"boolean"}
prompts_in_appendix = False #@param {type:"boolean"}

checks = {"References: each one found in a catalogue (or none)": references_found,
          "Quotations: each one checked in the original edition (or none)": quotations_checked,
          "My prompts copied in an appendix": prompts_in_appendix}
for label, done in checks.items():
    print("[x]" if done else "[ ] TO DO:", label)
print(f'\n"I used {tool} to {use}. My prompts are in the appendix. '
      f'Check: {check}. The analysis and the writing are my own."')

**What you should see** (with the default values)
```
[ ] TO DO: References: each one found in a catalogue (or none)
[ ] TO DO: Quotations: each one checked in the original edition (or none)
[ ] TO DO: My prompts copied in an appendix

"I used a chatbot (free online version, October 2026) to have the notion of temperature explained to me. My prompts are in the appendix. Check: I compared the answer with the chapter 3 lesson. The analysis and the writing are my own."
```
As long as a line says "TO DO", the declaration is not yet true: do the check, or remove that element from your work.

**Question:** which of the six rules in the checklist seems hardest to follow in your studies, and why?

## Your turn

Three changes to make with the fields, without writing any code:

1. **Enlarge the corpus.** In the first cell, type a sentence in `your_sentence` (for example "the teacher reads a novel in the morning"), then **Runtime → Run all**. Has the table for "the" changed? And the sentence for seed 2? Why?
2. **Inventing or copying?** In "generate a random sentence", try ten seeds (1 to 10). How many sentences are new (`False`)? How many make sense?
3. **Temperature.** In Part 2, move `temperature_sentences` from 0.1 to 3, in steps of 0.5, and each time write down the strangest sentence. Online, do the same experiment with the distilgpt2 slider.

## For the mini-project

The bigram model is an exploration tool: the most frequent pairs in a corpus reveal its habits and set phrases. Try it on your own data: readers' reviews (subject 1), open answers to a questionnaire (subject 2), political speeches (subject 3). Prepare a **CSV file in UTF-8** with a column called `text`, one sentence per line (a spreadsheet does this very well). In Colab, tick the box, run the cell and choose your file; in Jupyter, put the file `my_corpus.csv` next to the notebook.

**Before running:** as long as the box is not ticked, what will the cell do?

In [ ]:
#@title For the mini-project: your own corpus (optional)
load_my_file = False #@param {type:"boolean"}
word_to_study = "the" #@param {type:"string"}
import pandas as pd
if load_my_file:
    try:
        from google.colab import files
        file_name = list(files.upload())[0]     # Colab: choose your CSV file
    except ImportError:
        file_name = "my_corpus.csv"             # Jupyter: file placed next to the notebook
    table = pd.read_csv(file_name)              # a "text" column, one sentence per line
    my_model = count_next_words(list(table["text"].astype(str)))
    print(f'After "{word_to_study}":', my_model.get(word_to_study, Counter()).most_common(10))
else:
    print("No file loaded: tick the box, then run the cell again.")

**What you should see**
```
No file loaded: tick the box, then run the cell again.
```
With your file: the ten words that most often follow the studied word in your corpus, with their counts. For a real corpus, remove punctuation first, or accept that "city," and "city" count as two different words.

**Question:** in your corpus, which word would you like to study this way, and what do you hope to find?

## To hand in

Answer in writing, in English, in 8 to 12 lines for each question.

1. Copy a sentence produced by the bigram model (Part 1) that "sounds right" but means nothing. Using the pair-by-pair check, explain why the model produced it and what it lacks in order to make sense.
2. Compare the sentences obtained at $T = 0.2$ and at $T = 3$ (Part 2). Which temperature would you choose for a language exercise, and which one for a creative-writing workshop? Justify your choice with your own results.
3. Starting from the fake quotation in Part 4, explain to a classmate why a chatbot can invent a quotation or a reference. End with your own three checks and your declaration of use for the mini-project.

## Key points

- A language model predicts the next word from what it has seen. Our bigram model counts pairs of words; Transformers do the same with tokens, attention, a long context and millions of parameters.
- Temperature controls the draw: low, the text is predictable; high, it becomes surprising, then incoherent. Every pair can be right and the sentence absurd: plausible does not mean true.
- With a chatbot: find every reference, check every quotation in the original edition, declare any use.

Course lesson: [Chapter 3 · Deep learning and Transformers](https://progremer04.github.io/canva-au-for-ang-/en/#c3-deep) · Usage charter: [Generative AI in your classroom](https://progremer04.github.io/canva-au-for-ang-/en/enseignant.html#guide-ia)